### PyMongo
`PyMongo` è il driver ufficiale per l'utilizzo di MongoDB in applicativi Python e fornisce tutto il necessario per interagire con un'istanza MongoDB.

**Nota:** affinché il codice funzioni, *MongDB deve essere avviato*; le istruzioni per avviare MongoDB cambiano in base al sistema operativo, maggiori dettagli [qui](https://www.mongodb.com/docs/manual/administration/install-community/?operating-system=macos&macos-installation-method=tarball).

Ad esempio, su macOS basta fare (se MongoDB è stato installatato con Homebrew):
```zsh
brew services start mongodb-community@8.0
```

Chiaramente, essendo MongoDB un database non relazionale document-oriented, è utile tenere a mente la """gerarchia""":
```text
database -> collection -> documents
```

Il workdlow tipico è:

```python
from pymongo import MongoClient

with MongoClient("localhost", 27017) as client:
    db = client["test-database"]

    collection = db["test-collection"]
```

A questo punto, nulla è stato creato. `PyMongo` infatti usa una **creazione lazy**, cioè database e collection vengono creati automaticamente al primo inserimento di un documento. Per inserire uno o più documenti:

```python
post = {
    "author": "Mike",
    "text": "My first blog post!",
    "tags": ["mongodb", "python", "pymongo"]
}

# inserimento di un solo documento
collection.insert_one(post)

posts = [
    {
        "author": "Mike",
        "text": "My first blog post!",
        "tags": ["mongodb", "python", "pymongo"]
    },
    {
        "author": "Sara",
        "text": "Getting started with FastAPI",
        "tags": ["python", "fastapi", "rest"]
    },
    {
        "author": "Luca",
        "text": "Why I switched from SQL to MongoDB",
        "tags": ["mongodb", "database", "nosql"]
    },
    {
        "author": "Mike",
        "text": "Advanced aggregation pipelines in MongoDB",
        "tags": ["mongodb", "aggregation", "performance"]
    },
    {
        "author": "Anna",
        "text": "Building a REST API with Flask and PyMongo",
        "tags": ["flask", "pymongo", "python", "rest"]
    }
]

# inserimento di più documenti
collection.insert_many(posts)
```

**Nota:** Se un qualunque document (dizionario) inserito nella collection non presenta un campo _id, tale campo sarà aggiunto e popolato automaticamente.

Ora, una dimostrazione pratica della **lazy creation** di `PyMongo`; a tale scopo utilizziamo [`mongomock`](https://github.com/mongomock/mongomock), una libreria Python che implementa in memoria l'API di `PyMongo`, **simulando** il comportamento di un server **MongoDB** *senza richiederne uno reale.*

Semplicemente perché così questo test non "sporca" il DB, non scrive nulla sul disco, e di conseguenza dopo non c'è necessità di fare pulizia.

In [ ]:
import mongomock

with mongomock.MongoClient() as client:
    db = client["test-database"]

    print(client.list_database_names())

    collection = db["posts"]
    collection.insert_one({"author": "Mike"})
    print(client.list_database_names())
    print(db.list_collection_names())

In [ ]:
import pymongo

help(pymongo.database.Collection.insert_one)
help(pymongo.database.Collection.insert_many)

#### Ricerca
Come per l'inserimento, anche la **ricerca** prevede due funzioni distinte, una per la ricerca di un singolo documento e una per la ricerca di documenti multipli:

```python
# ricerca di un solo documento
# ritorna il primo documento che fa match con il filtro specificato
# oppure None in caso contrario
result = collection.find_one({"author": "Eliot"})

# ricerca documenti multipli

results = collection.find({"author": "Eliot"})
# ritorna un cursor per iterare sui risultati della query, cioè
# tutti i documenti che fanno match con il filtro specificato
for post in results:
    print(post["text"])
```

Per creare query più complesse, i filtri possono anche prevedere l'utilizzo di **operatori**:

| Operator | Meaning | Example | SQL Equivalent |
|----------|---------|---------|----------------|
| `$gt` | Greater Than | `"score":{"$gt":0}` | `>` |
| `$lt` | Less Than | `"score":{"$lt":0}` | `<` |
| `$gte` | Greater Than or Equal | `"score":{"$gte":0}` | `>=` |
| `$lte` | Less Than or Equal | `"score":{"$lte":0}` | `<=` |
| `$all` | Array Must Contain All | `"skills":{"$all":["mongodb","python"]}` | N/A |
| `$exists` | Property Must Exist | `"email":{"$exists":True}` | N/A |
| `$mod` | Modulo X Equals Y | `"seconds":{"$mod":[60,0]}` | `MOD()` |
| `$ne` | Not Equals | `"seconds":{"$ne":60}` | `!=` |
| `$in` | In | `"skills":{"$in":["c","c++"]}` | `IN` |
| `$nin` | Not In | `"skills":{"$nin":["php","ruby","perl"]}` | `NOT IN` |
| `$nor` | Nor | `"$nor":[{"language":"english"},{"country":"usa"}]` | N/A |
| `$or` | Or | `"$or":[{"language":"english"},{"country":"usa"}]` | `OR` |
| `$size` | Array Must Be Of Size | `"skills":{"$size":3}` | N/A |

In [ ]:
import pymongo

help(pymongo.database.Collection.find_one)
help(pymongo.database.Collection.find)

#### Cancellazione
Anche qui, due funzioni:

```python

# Rimozione di un solo documento in una collection
# cancella solo il primo documento che fa match con il filtro specificato
collection.delete_one({"author": "Eliot"})

# Rimozione di documenti multipli in una collection
# cancella 1+ documenti che matchano il filtro specificato
collection.delete_one({"author": "Eliot"})
```

In [ ]:
import pymongo

help(pymongo.database.Collection.delete_one)
help(pymongo.database.Collection.delete_many)

#### Aggiornamento
Anche qui, due funzioni:

```python

# Aggioramento di un solo documento in una collection
# aggiorna solo il primo documento che fa match con il filtro specificato
# la modifica è specificata nel secondo parametro (modificatore)
collection.update_one({"author": "Eliot"}, {"$set": {"author":"elliot"}})

# Rimozione di documenti multipli in una collection
# aggiorna 1+ documenti che matchano il filtro specificato
# la modifica è specificata nel secondo parametro (modificatore)
collection.update_many({"author": "Eliot"}, {"$set": {"author":"elliot"}})
```

Vari ***modificatori*** possono essere utilizzati con le operazioni di aggiornamento:

| Modifier | Meaning | Example |
|----------|---------|---------|
| `$inc` | Atomic Increment | `"$inc":{"score":1}` |
| `$set` | Set Property Value | `"$set":{"username":"niall"}` |
| `$unset` | Unset (delete) Property | `"$unset":{"username":1}` |
| `$push` | Atomic Array Append (atom) | `"$push":{"emails":"foo@example.com"}` |
| `$pushAll` | Atomic Array Append (list) | `"$pushall":{"emails":["foo@example.com","foo2@example.com"]}` |
| `$addToSet` | Atomic Append-If-Not-Present | `"$addToSet":{"emails":"foo@example.com"}` |
| `$pop` | Atomic Array Tail Remove | `"$pop":{"emails":1}` |
| `$pull` | Atomic Conditional Array Item Removal | `"$pull":{"emails":"foo@example.com"}` |
| `$pullAll` | Atomic Array Multi Item Removal | `"$pullAll":{"emails":["foo@example.com","foo2@example.com"]}` |
| `$rename` | Atomic Property Rename | `"$rename":{"emails":"old_emails"}` |


#### Thread-safety
Dalla docuementazione ufficiale:

> **Multithreading**
> 
> PyMongo is thread-safe and provides built-in **connection pooling for threaded applications**. Because each MongoClient object represents a pool of connections to the database, most applications require only a single instance of MongoClient, even across multiple requests.

Quindi, è opportuno creare una sola istanza `MongoClient` per processo, ed eventualmente passare quella ai thread. Un esempio di questo workflow è in `project/grpc_flask_mongoDB/grpc/server.py`

**Nota:** assunzione che **erroneamente** ho qualche volta fatto è che MongoDB salvi i documenti come stringa JSON, *introducento la necessità di convertire i numeri in stringhe* all'atto della query su campi numerici. In realtà, MongoDB internamente non salva JSON — salva **BSON** (Binary JSON), che è un formato binario che preserva i tipi nativi: *int, float, bool, date, ObjectId*, ecc.

#### Pulizia del database
Durante i test, o anche durante la prova d'esame se vogliamo, se non si ha la possibilità di instalare e usare `mongomock`, i seguenti comandi sono utili per capire cosa stia succedendo sotto il cofano e per fare un po' di pulizia:

```zsh
mongosh # per avviare la shell di MongoDB
```

```zsh
> show dbs # per mostrare i database
```

**Nota:** oltre i database creati dall'utente, vedrai i database di sistema creati automaticamente da MongoDB:

```text
admin
local
config
```

**Non toccarli!**

```zsh
> use <database_name> # per usare datababase_name (senza i < >!)
```

```zsh
> show collections # per mostrare le collections 
```

```zsh
> db.dropDatabase() # usando un certo database, per cancellarlo
```